# RasterioReader bytes-path knobs

`RasterioReader` exposes three keyword-only knobs that control how raster
bytes flow from storage into rasterio:

| Knob | Routes bytes through | Use when |
|---|---|---|
| _(default — no kwargs)_ | **GDAL VSI** (`libcurl` in C) | Default for cloud paths (`s3://`, `gs://`, `az://`, `https://`). Fastest sync option, no Python in the byte loop. |
| `opener=callable` | a user callback | Custom HTTP / auth, wrapping an async reader behind a sync facade, refreshable tokens. |
| `fs=fsspec.AbstractFileSystem` | **fsspec** | Niche backends GDAL doesn't speak (FTP, SFTP, GitHub, MinIO with custom endpoint), per-reader credential isolation. Shortcut for `opener=fs.open`. |

`opener=` and `fs=` are mutually exclusive — passing both raises `ValueError`.

This notebook demonstrates all three paths against a small **local fixture** so
it runs without credentials or network. The arithmetic and shapes are
identical across the three paths; only the byte transport changes.


## Setup — build a small local GeoTIFF fixture

In [1]:
import os
import tempfile

import numpy as np
import rasterio
from rasterio.transform import from_origin

tmpdir = tempfile.mkdtemp()
fixture_path = os.path.join(tmpdir, "demo.tif")

# 3-band, 64x64, deterministic content for round-trip checks
np.random.seed(0)
data = np.random.randint(0, 5000, size=(3, 64, 64), dtype=np.int16)

with rasterio.open(
    fixture_path, "w",
    driver="GTiff", height=64, width=64, count=3, dtype=data.dtype,
    crs="EPSG:32631", transform=from_origin(500000.0, 4600000.0, 10.0, 10.0),
    tiled=True, blockxsize=32, blockysize=32, compress="deflate",
) as dst:
    dst.write(data)

print(f"Fixture: {fixture_path}")


Fixture: /var/folders/k9/_v6ywhvj0nq36tpttd3j4mq80000gn/T/tmpepws9y1c/demo.tif


## Path 1 — Default (GDAL VSI)

No `opener=` / `fs=` / `rio_open_kwargs=` set. Rasterio handles the byte fetch
via GDAL's VSI layer. For cloud paths this is `libcurl` in C; for local files
it's just `fopen`. This is what every existing `RasterioReader(...)` call has
been doing.

In [2]:
from georeader.rasterio_reader import RasterioReader

reader_default = RasterioReader(fixture_path)

print("Internal state:")
print(f"  _opener            = {reader_default._opener}")
print(f"  _fs                = {reader_default._fs}")
print(f"  _rio_open_kwargs   = {reader_default._rio_open_kwargs}")
print(f"  _resolve_open_kwargs() = {reader_default._resolve_open_kwargs()}")
print()
data_default = reader_default.load().values
print(f"Loaded shape: {data_default.shape}, dtype: {data_default.dtype}")


Internal state:
  _opener            = None
  _fs                = None
  _rio_open_kwargs   = None
  _resolve_open_kwargs() = {}

Loaded shape: (3, 64, 64), dtype: int16


## Path 2 — `opener=callable`

Pass any callable matching `opener(path, mode='rb') -> file-like`. Rasterio
calls it for each byte range. Useful for custom HTTP clients, refresh-aware
token bearers, or a sync facade over an async reader.

> **Note on signature.** Rasterio 1.4+ calls `opener(path)` without an
> explicit `mode` argument, so the callable **must** have `mode='rb'` as a
> default. The `fs=` shortcut works because `fsspec`'s `fs.open(path, mode='rb')`
> already provides one.

In [3]:
def my_opener(path, mode="rb"):
    """Trivial opener that just returns a local binary file handle.

    In real use this could be a refresh-aware HTTP client, an obstore-backed
    sync facade, or anything else that emits a file-like object.
    """
    return open(path, "rb")

reader_opener = RasterioReader(fixture_path, opener=my_opener)

print("Internal state:")
print(f"  _opener            = {reader_opener._opener}")
print(f"  _resolve_open_kwargs() keys = {list(reader_opener._resolve_open_kwargs().keys())}")
print()
data_opener = reader_opener.load().values
print(f"Loaded shape: {data_opener.shape}")
print(f"Numerically identical to default path: {np.array_equal(data_opener, data_default)}")


Internal state:
  _opener            = <function my_opener at 0x1190e5300>
  _resolve_open_kwargs() keys = ['opener']

Loaded shape: (3, 64, 64)
Numerically identical to default path: True


## Path 3 — `fs=fsspec.AbstractFileSystem`

Shortcut equivalent to `opener=fs.open`. Use this when you're already
constructing an `fsspec` filesystem object — niche backends, custom auth,
per-reader credential isolation.

In [4]:
import fsspec

fs = fsspec.filesystem("file")
reader_fs = RasterioReader(fixture_path, fs=fs)

print("Internal state:")
print(f"  _fs                = {reader_fs._fs}")
print(f"  _resolve_open_kwargs() keys = {list(reader_fs._resolve_open_kwargs().keys())}")
print()
data_fs = reader_fs.load().values
print(f"Loaded shape: {data_fs.shape}")
print(f"Numerically identical to default path: {np.array_equal(data_fs, data_default)}")


Internal state:
  _fs                = <fsspec.implementations.local.LocalFileSystem object at 0x107532b90>
  _resolve_open_kwargs() keys = ['opener']

Loaded shape: (3, 64, 64)
Numerically identical to default path: True


## Validation — `opener=` and `fs=` are mutually exclusive

Passing both is almost certainly a bug, so the constructor raises `ValueError`
up front rather than silently picking one.

In [5]:
try:
    RasterioReader(fixture_path, opener=my_opener, fs=fs)
except ValueError as e:
    print(f"Caught: {e}")


Caught: RasterioReader: pass either `opener=` or `fs=`, not both. `fs=` is a shortcut for `opener=fs.open`.


## `rio_open_kwargs=` — escape hatch

Use `rio_open_kwargs=` to forward arbitrary additional keyword arguments to
every `rasterio.open(...)` call. This is the escape hatch for rasterio
options not surfaced as first-class kwargs (e.g. `sharing=False` to disable
the rasterio shared-dataset cache).

In [6]:
reader_extra = RasterioReader(fixture_path, rio_open_kwargs={"sharing": False})

print(f"_resolve_open_kwargs() = {reader_extra._resolve_open_kwargs()}")
print()
data_extra = reader_extra.load().values
print(f"Loaded shape: {data_extra.shape}")
print(f"Numerically identical to default path: {np.array_equal(data_extra, data_default)}")


_resolve_open_kwargs() = {'sharing': False}

Loaded shape: (3, 64, 64)
Numerically identical to default path: True


## Choosing a path

Mental model from the
[geostack ecosystem overview](https://github.com/jejjohnson/research_journal_v2/blob/main/notes/geotoolz/geostack.md):

- **Default (GDAL VSI)** — fastest for cloud reads. Use unless you have a
  specific reason not to.
- **`fs=`** — when you need a backend GDAL doesn't speak natively, or
  per-reader credential isolation (two `fsspec` filesystems with different
  auth in one process).
- **`opener=`** — full control. Custom HTTP clients, refresh-aware tokens,
  sync wrappers around async readers.

For high-concurrency *async* fan-out (tile servers, async ML services), the
right answer is usually a different reader entirely:
`georeader.AsyncGeoTIFFReader`, which routes through `obstore` and skips GDAL.


In [7]:
# Cleanup
import shutil
shutil.rmtree(tmpdir)
